# Bayesian Regression with Linear Basis Function Models

## Linear basis function models

Linear regression models are linear in their parameters, but not necessarily in their input variables. This lets them model any non-linearities from input variables to targets (eg. polynomial regression, shown later). 

A linear regression model y(x,w) can be defined as:

$y(x, w) = w_0 + \sum^{M-1}_{j=1} w_j{\phi}_j(x) = \sum^{M-1}_{j=0} w_j{\phi}_j(x) = w^T{\phi}(x)$

where ${\phi}_j$ are basis functions and $M$ is the number of parameters $w_j$ including the bias term $w_0$. 

Here we use the convention ${\phi}_0(x) = 1$.

In the simplest case of linear regression models, they also use linear functions from their input variables: the set of basis functions is the identity ${\phi}(x) = x$

The target variable $t$ of an observation $x$ is given by a deterministic function $y(x,w)$ plus adding random noise $\epsilon$:

$t = y(x,w) + \epsilon$

- We assume that the noise is normally distributed: has zero mean and precision (= inverse variance) $\beta$. The probabilistic model (conditional distribution of $t$ given $x$) is therefore:

$p(t | x,w,\beta) = N(t | y(x,w),{\beta}^{-1}) = \sqrt{{\beta}/2{\pi}} exp(-({\beta}/2)(t - y(x,w))^2)$

The mean of this distribution is the regression function $y(x,w)$

### Basically,

What that means is **the data is a curve plus some random wobble**.

- y(x,w) is the "true" smooth curve: the weighted sum of basis functions. It's deterministic: any x,w gives the same y.

- t is what you actually measure at that x: the dot in the scatter plot.

- ${\epsilon}$ is why the dot doesn't sit exactly on the curve -- measurements are messy.

So dot = curve + wobble.

"Noise is normally distributed with zero mean" means the wobble follows a bell curve centered on 0. Dots are equally likely to be above/below the curve. No bias. Normally distributed means small wobbles are common and big ones are rare.

Precision $\beta$ is how tight the wobble is:
- variance ${\sigma}^2$ = how spread out the noise is
- precision ${\beta} = 1/{\sigma}^2$ = how concentrated the noise is

Big $\beta$ means the dots hug the curve; small means they scatter widely. Precision is used because it makes the algebra tighter; $\beta^{-1}$ = variance.

**The "conditional distribution of t given x"**, $p(t | x,w,{\beta})$: *if I already know the input x, weights w, and noise level ${\beta}$, how likely is each posible measured value t?*

- Since t = curve + bell-shaped noise, the answer is a bell curved centered on y(x,w) with variance $1/\beta$.
- The big formula is the standard bell-curve (Gaussian) formula with those values plugged in. It says "values of t near y(x,w) are likely, and values far are unlikely.

If you measured the same x many times, your dots would average out to y(x,w). The curve is your best guess, and the bell describes the scatter around it.

This setup matters for the next parts, because it lets you score a candidate w: "given the curve had these weights, how likely are the dots I saw?" = the **likelihood**. And then comes the Bayesian step: combine it with a **prior** (belief about w before seeing data) to get a **posterior** (beleif after seeing).

## Likelihood function

**Remember the big question** we're trying to answer:

we have some noisy measurements, and we want to know which curve produced them, plus how sure we can be about that.

Say we're tracking how an apartment's temperature (t) changes over time of day (x) (hours). We take 10 readings $x_1,...,x_{10}$ : $t_1,...,t_{10}$.
- $X$ is all our inputs stacked together: the 10 hours
- $t$ is all our measurements: the 10 temperatures.

We believe there's a smooth daily temperature curve, so each reading is that curve plus a bit of thermometer wobble. The curve is set by the weights $w$. **we never get to see $w$; we only see the dots**.

So in this section, we want to work backwards from the dots to get $w$.

- The **likelihood** answers one question: **"If the weights were w, how plausible would our 10 readings be?"**
    - Try a w. Let's say that w's curve passes close to all 10 dots. Each would need a small, typical wobble (to touch the curve), so the readings are plausible and we conclude likelihood is high.
    - Try a w. Let's say its curve sits 5 degrees above everything. Every dot would need a big, rare wobble to touch the curve. So likelihood is low.

Likelihood therefore works like a **scoring function for candidate curves**: how well does this curve explain what we saw?

-----

For model fitting and model parameter inference, we use a training set of N independent, identically distributed (i.i.d) observations: $x_1,...,x_N$ and corresponding targets: $t_1,...,t_N$.

We combine the column vectors $x_i$ into matrix X, where $X_{i,:} = x^T_i$, and scalar targets $t_i$ into column vector $t$.

The joint conditional probability of targets $t$ given $X$ can be formulated as:

$p(t|X,w,\beta) = \prod_{i=1}^N \mathcal{N}(t_i | w^T \phi(x_i),{\beta}^{-1})$

For better readability, this function will be written as $p(t | w,\beta)$ instead of $p(t|X,w,\beta)$ from now on. The log likelihood of this function:

$\log p(t|w,\beta) = N/2 \log \beta - N/2 \log 2\pi - \beta E_D(w)$

- Where E_D(w) is the sum-of-squares error function coming from the exponent of the likelihood function:

$E_D(w) = 1/2 \sum^N_{i=1} (t_i - w^T \phi (x_i))^2 = 1/2 ||t-\boldsymbol\Phi w||^2$

- Matrix $\boldsymbol\Phi$ is the *"design matrix"* (NxM basis function matrix), defined as

$$
\boldsymbol\Phi = 
\begin{pmatrix}
\phi_0(\mathbf{x}_1) &  \phi_1(\mathbf{x}_1) & \cdots & \phi_{M-1}(\mathbf{x}_1) \\ 
\phi_0(\mathbf{x}_2) &  \phi_1(\mathbf{x}_2) & \cdots & \phi_{M-1}(\mathbf{x}_2) \\
\vdots & \vdots & \ddots & \vdots \\
\phi_0(\mathbf{x}_N) &  \phi_1(\mathbf{x}_N) & \cdots & \phi_{M-1}(\mathbf{x}_N)
\end{pmatrix} \tag{7}
$$


Simpler:

1. For each reading i, compute the bell-curve score (last section): how close is $t_i$ to the curve's height at $x_i$?
2. Multiply the N scores together. The noise is assumed independent across readings, so they just multiply; a product is high only if *every* dot fits reasonably well.
3. Taking the log turns the **product** into a **sum**, and the only part that depends on w is $-\beta E_D(w)$. So the w with the highest likelihood is the one with smallest sum of squared errors. Maximum likelihood here is least squares.

After this section... 
- **Non-bayesian route**: we'll be able to pick the single w with the highest score (maximum likelihood; here ordinary least squares). We'll get on best curve.
- **Bayesian route (this notebook)**: don't pick just one, but combine the score with our **prior** to get the **posterior**, a probability for every possible w. We'll end up with a best-guess curve plus an uncertainty band. With the thermometer example, this would be confident around the hours we measured and say "not sure" at a time where have no readings. 

That uncertainty band is the whole payoff we're building towards.

Pipeline of what we do:

prior over w → (× likelihood of the readings) → posterior over w → (average over curves) → distribution of temperature at any hour

==

combine the likelihood for all candidate ws with our prior over w to get a posterior over w. Then use that to get the distribution of temperatures at any given hour.

## Maximum likelihood 

Maximizing log likelihood (= minimizing the sum-of-squares error function) wrt **w** gives the maximum likelihood estimate of parameters **w**. 

Maximum likelihood estimation can lead to severe over-fitting if complex models are fit to datasets of limited size. A common approach to prevent this is adding a regularization term (seen in makemore section) over the error function. We will see shortly that **this regularization term arises naturally** when following a Bayesian approach (when defining a prior distribution over parameters w).

## Bayesian approach

Now let's do the bayesian approach.

### Prior and posterior distribution

For a Bayesian treatment of linear regression, we need a prior probability distribution over model parameters $w$. For simplicity, we will use an isotropic Gaussian distribution (def below) over parameters $w$ with zero mean. Prior:

$$p(\mathbf{w} \lvert \alpha) = \mathcal{N}(\mathbf{w} \lvert \mathbf{0}, \alpha^{-1}\mathbf{I}) \tag{8}$$

An isotropic Gaussian distribution has a diagonal covariance matrix where all diagonal elements have the same variance $\alpha^{-1}$ ($\alpha$ is the precision of the prior). 

This formula says "before seeing any data, I think each weight is probably near 0, and I'm equally unsure about all of them in the same way."

A **covariance matrix** is a bell curve in many dimensions. With one number we only need a variance (how spread out our guess is). But with several ws ($w_0,...$) we need to describe two things:
1. How unsure are we about each weight on its own? These are the variances on the diagonal of the matrix.
2. Do the weights move together? For example, "if $w_1$ is big, $w_2$ is probably big". These are the covariances, off the diagonal.

Isotropic means "the same in every direction". An isotropic Gaussian has 
- the same variance for every weight
- zero covariance between weights.

### Now I will fast forward the math derivation sections and get to the meat of things (coding implementation)

Here are the important four equations:

1. Posterior precision: $S^{-1}_N = \alpha I + \beta \Phi^T\Phi$
2. Posterior mean: $m_N = \beta S_N \Phi^T t$
3. Predictive mean at a new x: $m^T_N \phi (x)$
4. Predictive variance at a new x: $1/\beta + \phi (x)^T S_N \phi (x)$

In plain terms:

1. Posterior precision $(S^{-1}_N)$ = prior certainty ($\alpha$; $1/\alpha$ is the prior variance) plus certainty from the data ($\beta, \Phi$) = how sure we are about $w$.
2. Posterior mean $m_N$ = the best-guess weights.
3. Predictive mean at a new x = the best-guess curve.
4. Predictive variance at a new x = noise (thermometer wobble) plus our uncertainty about the curve.

## Implementation

### Posterior and posterior predictive distribution

`posterior` computes the mean and covariance matrix of the posterior distribution. Given the readings, update the believed weights.
- In: 
    - the design matrix $\Phi$ (inputs passed through basis functions, one row per reading)
    - the measurements t
    - the two "confidence" knobs: $\alpha$ (prior), $\beta$ (noise)
- Out:
    - `m_N` (best-guess weights)
    - `S_N` (how unsure we still are about them)

`posterior_predictive` computes the mean and the variances of the posterior predictive distribution. Given new inputs (eg, an hour of the day where you have no readings), determine expected temprature and how sure we are.
- In: 
    - $\Phi$ for the *test* inputs (often a fine grid of x for plotting)
    - outputs of function `posterior` function
- Out: 
    - for each test x, a **mean** (curve's height) and a **variance** (how wide the band is there).

These functions together turn **dots -> weights -> curve with a band.**



In [ ]:
import numpy as np


def posterior(Phi, t, alpha, beta, return_inverse=False):
    """Computes mean and covariance matrix of the posterior distribution"""
    S_N_inv = alpha * np.eye(Phi.shape[1]) + beta * Phi.T.dot(Phi)
    S_N = np.linalg.inv(S_N_inv)
    m_N = beta * S_N.dot(Phi.T).dot(t)
    
    if return_inverse:
        return m_N, S_N, S_N_inv
    else:
        return m_N, S_N
    
    
def posterior_predictive(Phi_test, m_N, S_N, beta):
    """Computes mean and variances of the posterior predictive distribution"""
    y = Phi_test.dot(m_N)
    # Only compute variances (diagonal elements of covariance matrix)
    y_var = 1 / beta + np.sum(Phi_test.dot(S_N) * Phi_test, axis=1)
    
    return y, y_var

### Example datasets

In [ ]:
f_w0 = -0.3
f_w1 =  0.5


def f(X, noise_variance):
    '''Linear function plus noise'''
    return f_w0 + f_w1 * X + noise(X.shape, noise_variance)


def g(X, noise_variance):
    '''Sinusoidial function plus noise'''
    return 0.5 + np.sin(2 * np.pi * X) + noise(X.shape, noise_variance)


def noise(size, variance):
    return np.random.normal(scale=np.sqrt(variance), size=size)

### Basis functions

In [ ]:
def identity_basis_function(x):
    return x


def gaussian_basis_function(x, mu, sigma=0.1):
    return np.exp(-0.5 * (x - mu) ** 2 / sigma ** 2)


def polynomial_basis_function(x, power):
    return x ** power


def expand(x, bf, bf_args=None):
    if bf_args is None:
        return np.concatenate([np.ones(x.shape), bf(x)], axis=1)
    else:
        return np.concatenate([np.ones(x.shape)] + [bf(x, bf_arg) for bf_arg in bf_args], axis=1)

### Straight line fitting

For straight-line fitting, we use a linear regression model with only $w_0$, $w_1$. Predictions are made with `posterior_predictive` distribution. Since this model only has two parameters, we can visualize the posterior density in 2D, which is done in the first column after the output. Rows use increasing number of training data from a dataset.

In [ ]:
np.random.seed(0)

from reference.bayesian_linear_regression_util import *

import matplotlib.pyplot as plt
%matplotlib inline

# Training dataset sizes
N_list = [1, 3, 20]

beta = 25.0 # Noise
alpha = 2.0 # Prior

# Training observations in [-1, 1)
X = np.random.rand(N_list[-1], 1) * 2 - 1

# Training target values
t = f(X, noise_variance=1/beta)

# Test observations
X_test = np.linspace(-1, 1, 100).reshape(-1, 1)

# Function values without noise
y_true = f(X_test, noise_variance=0)

# Design matrix of test observations
Phi_test = expand(X_test, identity_basis_function)

plt.figure(figsize=(15, 10))
plt.subplots_adjust(hspace=0.4)

for i, N in enumerate(N_list):
    X_N = X[:N]
    t_N = t[:N]
    
    # Design matrix of training observations
    Phi_N = expand(X_N, identity_basis_function)
    
    # Mean and covariance matrix of posterior
    m_N, S_N = posterior(Phi_N, t_N, alpha, beta)
    
    # Mean and variances of posterior predictive
    y, y_var = posterior_predictive(Phi_test, m_N, S_N, beta)
    
    # Draw 5 random weight samples from posterior and compute y values
    w_samples = np.random.multivariate_normal(m_N.ravel(), S_N, 5).T
    y_samples = Phi_test.dot(w_samples)
    
    plt.subplot(len(N_list), 3, i * 3 + 1)
    plot_posterior(m_N, S_N, f_w0, f_w1)
    plt.title(f'Posterior density (N = {N})')
    plt.legend()
    
    plt.subplot(len(N_list), 3, i * 3 + 2)
    plot_data(X_N, t_N)
    plot_truth(X_test, y_true)
    plot_posterior_samples(X_test, y_samples)
    plt.ylim(-1.5, 1.0)
    plt.legend()
    
    plt.subplot(len(N_list), 3, i * 3 + 3)
    plot_data(X_N, t_N)
    plot_truth(X_test, y_true, label=None)
    plot_predictive(X_test, y, np.sqrt(y_var))
    plt.ylim(-1.5, 1.0)
    plt.legend()

### Gaussian basis functions examples

How to fit a Gaussian basis function model to a noisy sinusoidal dataset. We now use 9 Gaussian basis functions and thusly infer 10 parameters instead of 2. We therefore cannot display the posterior density unless we just take 2 of those 10 parameters randomly.

In [ ]:
N_list = [3, 8, 20]

beta = 25.0 # noise
alpha = 2.0 # prior

# Training observations in [0, 1)
X = np.random.rand(N_list[-1], 1)

# Training target values
t = g(X, noise_variance=1/beta)

# Test observations
X_test = np.linspace(0, 1, 100).reshape(-1, 1)

# Function values without noise
y_true = g(X_test, noise_variance=0)

# Design matrix of test observations
Phi_test = expand(X_test, bf=gaussian_basis_function, bf_args=np.linspace(0, 1, 9))

plt.figure(figsize=(10, 10))
plt.subplots_adjust(hspace=0.4)

for i, N in enumerate(N_list):
    X_N = X[:N]
    t_N = t[:N]
    
    # Design matrix of training observations
    Phi_N = expand(X_N, bf=gaussian_basis_function, bf_args=np.linspace(0, 1, 9))
    
    # Mean and covariance matrix of posterior
    m_N, S_N = posterior(Phi_N, t_N, alpha, beta)
    
    # Mean and variances of posterior predictive
    y, y_var = posterior_predictive(Phi_test, m_N, S_N, beta)
    
    # Draw 5 random weight samples from posterior and compute y values
    w_samples = np.random.multivariate_normal(m_N.ravel(), S_N, 5).T
    y_samples = Phi_test.dot(w_samples)
    
    plt.subplot(len(N_list), 2, i * 2 + 1)
    plot_data(X_N, t_N)
    plot_truth(X_test, y_true)
    plot_posterior_samples(X_test, y_samples)
    plt.ylim(-1.0, 2.0)
    plt.legend()
    
    plt.subplot(len(N_list), 2, i * 2 + 2)
    plot_data(X_N, t_N)
    plot_truth(X_test, y_true, label=None)
    plot_predictive(X_test, y, np.sqrt(y_var))
    plt.ylim(-1.0, 2.0)
    plt.legend()

Basically. Now instead of using slope + intercept to model a straight line, it models a curve with 9 Gaussian bumps, + a bias. The hidden truth is a sine wave (black dashed line).

- Rows go 3 -> 8 -> 20 dots. 
- Left column shows 5 plausible curves (red) sampled from the posterior.
- Right column shows the best guess plus the uncertainty band (affected by noise $\beta$ and prior certainty $\alpha$).

**Main takeaway**

Uncertainty is local: with the straight line, one dot fixed the whole line fairly well; with bumps, each bump only affects its own neihbourhood. *The band is literally a map of where we have data.*

## That completes

the Krasser BLR ipynb part of this unit. Parking that for now. 

Now moving to the worksheet part: steps 1-8. These involve running a version of the above in a toy world, made to better fit my thesis study.

First, a translation table, from the language above into language for the thesis.
- a look (x, y): one data point.
- the map: the posterior. What we currently believe about the curve.
- A: $S_N^{-1}$: posterior precision. How sure we are about the weights, starting at $\alpha I$ (prior only)
- b: $\beta \Phi^T t$. Running total of evidence from readings.
- $m = A^{-1}b$: $m_N$. best-guess weights.
- $\mu (x)$: predictive mean. best-guess curve.
- $\sigma _f(x)$: $\sqrt (\phi^T S_N \phi)$. uncertainty about the *curve* without the noise term
- $\sigma_n = 0.1$: $1/\sqrt\beta$: $\beta = 100$. noise / wobble.
- lattice: X_test. 101 x-values to plot on.
- $\epsilon^2$: (new). "how surprised was I?" -- visited on step 6.

The only real difference is the notebook builds A and b at once from $\phi$. The worksheet also lets us build them one reading at a time by adding to them. Step 5 checks both approaches give the same answer.

In **the toy world**, the true curve is a single bump with y=1 at x=0.7; flat 0 everywhere else. We will model it with 20 Gaussian bumps (no bias column this time) spread evenly over 0-1.

### 1. Features. Write Phi(x): one row per x, one column per feature.

*Predict first: The shape of Phi([0.5]), and which two columns are largest.*

Phi(x) are the basis functions: the Gaussian bumps each input x is read through. 

- Each Phi(x) should take one or more locations (eg. [0.5]) or the whole lattice (101 points).
- Each Phi(x) should output a table with one row per x and one column per bump (feature).
    - Entry (i, j) = how much bump j responds at location $x_i$

$$\phi_j(x) = \exp\left(-\frac{(x-c_j)^2}{2\cdot 0.07^2}\right)$$

Prediciton first.
1. What shape will `Phi([0.5])` be?
2. Which two columns will be largest, and why two rather than one?

Answer:
- Phi([0.5]) will be shape 1x20
- The two largest columns will be will be the center two. The two bumps in between 0-1 will be largest, as they are closest to 0.5. Two columns instead of one because 20 is even, and columns 9 and 10 are equally far from it.

In [ ]:
import numpy as np

centres = np.linspace(0, 1, 20) # points centres
width = 0.07 # gaze bump spread across points: looking at one point updates neighbours

def Phi(x):
    x = np.asarray(x, dtype=float).reshape(-1, 1) # make x a column: shape (N, 1)
    d = x - centres                               # (N, 1) - (20,) -> (N, 20)
    return np.exp(-d**2 / (2 * width**2))         # bump height for every pair

# Check
np.set_printoptions(suppress=True)
print(Phi([0.5]).shape)
print(np.round(Phi([0.5]), 4))
print(Phi([0.5]).sum().round(4)) # row sum (should be 3.3)
print(Phi([0.5]).argsort()[0, -2:]) # indices of the two largest columns

plt.plot(np.linspace(0, 1, 101), Phi(np.linspace(0, 1, 101)));

### Step 2: Prior band. With no data, $\sigma_f(x)$ is the sqrt of the sum of $\phi_j(x)^2$ over $\alpha$. Plot it across the lattice.

*predict first: is the band flat from 0 to 1?*

Prediction:
- Initially I would think the prior band should be flat: The model is equally uncertain about what's at every point, at initialization.
- However thinking more, the band should actually be higher in the centre (peaking flat around 0.2 - 0.8), and lower on the edges; see the graph above. There are less bumps on the sides; less overlapping bumps, less uncertainty.

$$m = A^{-1}b, \qquad \mu(x) = \phi(x)^\top m, \qquad \sigma_f^2(x) = \phi(x)^\top A^{-1}\phi(x)$$

In [ ]:
alpha = 1.0 # prior
lattice = np.linspace(0, 1, 101) # 101 point to read out
P = Phi(lattice) # (101, 20): every lattice point read through every bump

# with no data, A = alpha*I, so A^-1 = I/alpha and 
# phi^T A^-1 phi = sum_j phi_j(x)^2 / alpha
sigma_f_prior = np.sqrt((P**2).sum(axis=1) / alpha)

# Same thing via general formula; we'll reuse from step 3 onwards
A = alpha * np.eye(20)
A_inv = np.linalg.inv(A)
sigma_f_general = np.sqrt(np.sum(P @ A_inv * P, axis=1)) # tutorial's diagonal trick
print(np.allclose(sigma_f_prior, sigma_f_general))       # should be True

for x in [0.0, 0.2, 0.5, 0.7, 1.0]:
    i = int(round(x * 100))
    print(x, sigma_f_prior[i].round(4))
    
plt.plot(lattice, sigma_f_prior)
plt.plot(lattice, 0 * lattice, 'k--', label='prior mean mu = 0')
plt.fill_between(lattice, -2*sigma_f_prior, 2*sigma_f_prior, alpha=0.2, label='±2σ_f')
plt.xlabel('x (location in scene)'); plt.legend();

### Step 3: Batch posterior. 

Five looks at x = 0.10, 0.35, 0.60, 0.70, 0.90 returned saliency readings y = 0.02, -0.05, 0.41, 0.93, 0.04. Build A and b in one go. Plot the mean and a band of two $\sigma_f$

*predict first: After one look at 0.7, is the band there nearer the prior's 1.5 or the noise's 0.1?*

$$A = \alpha I + \frac{\Phi^T\Phi}{\sigma_n^2}, \qquad b = \frac{\Phi^T y}{\sigma_n^2}$$

Prediction answer.
- Before the gaze, the agent is very unsure about 0.7 (prior ~1.5; wide band). Then it looked and measured it with a noisy sensor (noise 0.1). The question is how unsure should it be right after measuring that exact spot? 
- It should be roughly as unsure as the sensor is noisy. The ignorance-uncertainty factor goes down (because it looked there; it's no longer ignorant), but it still can't be fully sure, because there is noise. So the $\sigma_f$ drops to roughly $\sigma_n$. More looks would shrink it more. 

*Prediction 2:*
- *At 0.65 (near a gaze, but not on one): is the band small, large, or somewhere between?*
- *At 0.2 and 0.5 (no gazes nearby): what does the band look like there?*

Prediction 2 answer.
- At 0.65, the band should be smaller than pre-gaze, but bigger than at direct gaze point 0.7. 
- If we're assuming the model also looked at 0.10, 0.35, 0.60, the band for 0.2 and 0.5 should be lower than the prior (since they're close enough to gazes 0.10 and 0.60 and therefore share some bump), but bigger than those points directly.

In [ ]:
sigma_n = 0.1 # noise
alpha = 1.0   # prior

# The five gazes
x_obs = np.array([0.10, 0.35, 0.60, 0.70, 0.90])
y_obs = np.array([0.02, -0.05, 0.41, 0.93, 0.04])

# Build the posterior in one go
Phi_obs = Phi(x_obs)                                      # (5, 20): each gaze read through every bump
A = alpha * np.eye(20) + Phi_obs.T @ Phi_obs / sigma_n**2 # certainty about the weights
b = Phi_obs.T @ y_obs / sigma_n**2                        # accumulated evidence

# Read the map on the lattice
A_inv = np.linalg.inv(A)
m = A_inv @ b                                    # best-guess weights
mu = P @ m                                       # best-guess saliency everywhere
sigma_f = np.sqrt(np.sum(P @ A_inv * P, axis=1)) # uncertainty everywhere (no noise)

# Compare with the worksheet
for x in [0.2, 0.5, 0.65, 0.7, 0.9]:
    i = int(round(x * 100))
    print(f'x={x}: mu={mu[i]:.4f}  sigma_f={sigma_f[i]:.4f}')

# Plot
true_f = np.exp(-(lattice - 0.7)**2 / (2 * 0.08**2)) # the real scene
plt.plot(lattice, true_f, 'k--', label='true saliency')
plt.plot(lattice, mu, label='belief mu')
plt.fill_between(lattice, mu - 2*sigma_f, mu + 2*sigma_f, alpha=0.2, label='±2σ_f')
plt.scatter(x_obs, y_obs, c='k', zorder=3, label='gazes')
plt.xlabel('x (location in scene)'); plt.legend();

The band pinches at each of the 5 gazes, and swells between them. The mean hugs zero away from 0.6-0.7, and rises to nearly 1 around 0.7.

### Step 4: Ladder link

Set $\alpha$ to 1e-6 and compare the weights with np.linalg.lstsq (plain least squares -- bayesian with no prior = ordinary least squares).

*Predict first: Does the mean now pass through all five points? With $\alpha$ = 1, why is $\mu(0.7)$ below 0.93?*

Prediction answer.
- Setting the **prior's precision** $\alpha$ to tiny. Prior precision acts as a regularizer. Big $\alpha$ means a strong pull, heavy regularization, smoother curve, more dragged to 0. Tiny $\alpha$ will cause almost no pull/regularization, which means the data will be trusted strongly. So the band should be mega pinched to the points (noise floor = 0.1), and explodes between readings. The mean should pass through the points very closely.
- Why is $\mu$(0.7) below 0.93 with $\alpha = 1$? Because the prior is pulling it down towards 0 ($\alpha$ is the prior precision = pull factor) from the observed y = 0.93. It sees a noisy 0.93, expects it to usually be closer to 0, so assumes it's within some noise below 0.93.

In [ ]:
# Step 4: nearly switch off the prior
alpha_tiny = 1e-6
A_tiny = alpha_tiny * np.eye(20) + Phi_obs.T @ Phi_obs / sigma_n**2
m_tiny = np.linalg.solve(A_tiny, b) # same b as step 3: the evidence hasn't changed

# Ordinary least squares on the same five gazes
w_lstsq = np.linalg.lstsq(Phi_obs, y_obs, rcond=None)[0]

print('max weight difference:', np.abs(m_tiny - w_lstsq).max()) # worksheet: ~2e-8
print('fit at the gazes:', (Phi_obs @ m_tiny).round(4))         # should equal y_obs
print('readings:        ', y_obs)

# Plot: mean only (the band is ~1000 between gazes and would flatten the plot)
mu_tiny = P @ m_tiny
plt.plot(lattice, true_f, 'k--', label='true saliency')
plt.plot(lattice, P @ m, label='α = 1')
plt.plot(lattice, mu_tiny, label='α = 1e-6')
plt.scatter(x_obs, y_obs, c='k', zorder=3, label='gazes')
plt.xlabel('x (location in scene)'); plt.legend();

alpha = 1.0   # back to the worksheet's prior for step 5

### Step 5: Sequential

Start from A = αI and b = 0, and add the five looks one at a time.

*Predict first: Does the order of the looks change the final map?*

Prediction answer.
- Each gaze just adds something to A and b. Sums don't care about order. So the final map should be the same, whether "one at a time" or "all at once". They're the same sum, computed differently.
- This of course will change once we add forgetting.

In [ ]:
def update(A, b, x, y):
    """Add one gaze (x, y) to the map."""
    phi = Phi([x])[0]                       # (20,): this location read through every bump
    A = A + np.outer(phi, phi) / sigma_n**2 # "φφᵀ/σₙ²": certainty gained from looking here
    b = b + phi * y / sigma_n**2            # "φy/σₙ²": evidence gained from what was seen
    return A, b

def fresh_map():
    return alpha * np.eye(20), np.zeros(20) # prior only: A = αI, b = 0

# One gaze at a time, in the original order
A_seq, b_seq = fresh_map()
for x, y in zip(x_obs, y_obs):
    A_seq, b_seq = update(A_seq, b_seq, x, y)
    
# Compare with step 3's batch A and b
print('A diff:', np.abs(A_seq - A).max()) # worksheet: ~1e-12
print('b diff:', np.abs(b_seq - b).max())

# Now let's shuffle the order a few times
rng = np.random.default_rng(0)
for trial in range(3):
    order = rng.permutation(5)
    A_s, b_s = fresh_map()
    for k in order:
        A_s, b_s = update(A_s, b_s, x_obs[k], y_obs[k])
    print(order, np.abs(A_s - A).max(), np.abs(b_s - b).max())

### Step 6: Surprise

Compute $\epsilon^2$ before each of those five updates.

*Predict first: the fourth look misses by 0.68. Is its $\epsilon^2$ above or below 1? Then a sixth look at 0.7 returns 0.50, a miss of only 0.43: guess its $\epsilon^2$*

Prediction answer.
- abcabc for next time